# 24 — Estudio Comparativo de Reducción Dimensional Multi-Backbone: PCA vs. Linear vs. S-PCA


---

## Objetivo

Este cuaderno compara sobre cómo reducir de $D \to 64$ dimensiones en extractores convolucionales de alta dimensión ($D = 512, 1024, 1280, 2048$) evaluando tres paradigmas matemáticos:

1. `PCA` (No supervisado, ortogonal): Descomposición espectral clásica sin etiquetas.
2. `lin` (Supervisado, no ortogonal): Proyección lineal entrenada con entropía cruzada hacia el manifold patológico.
3. `S_PCA` (Supervisado + Ortogonal): Composición en 2 etapas: proyección supervisada $D \to 64$ seguida de rotación espectral (SVD/PCA).

### Matriz Experimental (12 combinaciones):
$$\{ \text{ResNet18 (512D)}, \text{DenseNet121 (1024D)}, \text{EfficientNet-B0 (1280D)}, \text{ResNet50 (2048D)} \} \times \{ \text{PCA}, \text{Linear}, \text{S-PCA} \}$$


In [ ]:
import os
import sys
import time
import json
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))   # raiz del repo (ejecutar desde pipeline/)
CODIGOS = PROJECT_ROOT
REP = os.path.join(PROJECT_ROOT, "artifacts")
for p in (CODIGOS, REP, os.path.join(CODIGOS, "src")):
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

from src.explainability.f2_symb import SymbHead

EMB_DIR = os.path.join(REP, "output", "embeddings")
REPORTS_DIR = os.path.join(CODIGOS, "reports", "experiments")
FIGS_DIR = os.path.join(CODIGOS, "reports", "figures")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo activo: {DEVICE} | PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Dispositivo activo: cuda | PyTorch: 2.14.0.dev20260810+cu130
GPU: NVIDIA GeForce RTX 5060


In [ ]:
BACKBONES = ["resnet18", "densenet121", "efficientnet_b0", "resnet50"]
METHODS = ["PCA", "Linear", "S-PCA"]

embeddings_data = {}
for b in BACKBONES:
    p = 'f2base' if b == 'resnet18' else f'f2_{b}'
    tr = np.load(os.path.join(EMB_DIR, f"{p}_train_feats.npy"))
    va = np.load(os.path.join(EMB_DIR, f"{p}_val_feats.npy"))
    te = np.load(os.path.join(EMB_DIR, f"{p}_test_feats.npy"))
    y_tr = np.load(os.path.join(EMB_DIR, f"{p}_train_labels.npy"))
    y_va = np.load(os.path.join(EMB_DIR, f"{p}_val_labels.npy"))
    y_te = np.load(os.path.join(EMB_DIR, f"{p}_test_labels.npy"))
    embeddings_data[b] = {"X_tr": tr, "X_va": va, "X_te": te, "y_tr": y_tr, "y_va": y_va, "y_te": y_te}
    print(f"{b:16s}: Train {tr.shape}, Val {va.shape}, Test {te.shape}")

resnet18        : Train (4278, 512), Val (1134, 512), Test (1012, 512)
densenet121     : Train (4278, 1024), Val (1134, 1024), Test (1012, 1024)
efficientnet_b0 : Train (4278, 1280), Val (1134, 1280), Test (1012, 1280)
resnet50        : Train (4278, 2048), Val (1134, 2048), Test (1012, 2048)


In [ ]:
# Definición de los 3 reductores en estudio
class PCAReducer:
    def __init__(self, out_dim=8, random_state=42):
        self.pca = PCA(n_components=out_dim, random_state=random_state)
    def fit(self, X_tr):
        z = self.pca.fit_transform(X_tr).astype(np.float32)
        self.mu = z.mean(0, keepdims=True); self.sd = z.std(0, keepdims=True) + 1e-8
        return self
    def transform(self, X):
        z = self.pca.transform(X).astype(np.float32)
        return (z - self.mu) / self.sd

class LinearBottleneckReducer:
    def __init__(self, in_dim, out_dim=8, bottleneck_dim=64):
        self.proj = nn.Linear(in_dim, bottleneck_dim).to(DEVICE)
        self.head = nn.Linear(bottleneck_dim, 2).to(DEVICE)
        self.out_dim = out_dim
    def fit(self, X_tr, y_tr, epochs=25):
        opt = torch.optim.Adam(list(self.proj.parameters()) + list(self.head.parameters()), lr=1e-3, weight_decay=1e-3)
        crit = nn.CrossEntropyLoss()
        t_X = torch.from_numpy(X_tr).float().to(DEVICE); t_y = torch.from_numpy(y_tr).long().to(DEVICE)
        for ep in range(epochs):
            opt.zero_grad(); crit(self.head(self.proj(t_X)), t_y).backward(); opt.step()
        with torch.no_grad(): z = self.proj(t_X).cpu().numpy()
        self.top = np.argsort(z.var(axis=0))[::-1][:self.out_dim]
        z_sub = z[:, self.top]
        self.mu = z_sub.mean(0, keepdims=True); self.sd = z_sub.std(0, keepdims=True) + 1e-8
        return self
    def transform(self, X):
        with torch.no_grad(): z = self.proj(torch.from_numpy(X).float().to(DEVICE)).cpu().numpy()
        return (z[:, self.top] - self.mu) / self.sd

class SupervisedPCAReducer:
    def __init__(self, in_dim, out_dim=8, bottleneck_dim=64):
        self.proj = nn.Linear(in_dim, bottleneck_dim).to(DEVICE)
        self.head = nn.Linear(bottleneck_dim, 2).to(DEVICE)
        self.pca = PCA(n_components=out_dim, random_state=42)
    def fit(self, X_tr, y_tr, epochs=25):
        opt = torch.optim.Adam(list(self.proj.parameters()) + list(self.head.parameters()), lr=1e-3, weight_decay=1e-3)
        crit = nn.CrossEntropyLoss()
        t_X = torch.from_numpy(X_tr).float().to(DEVICE); t_y = torch.from_numpy(y_tr).long().to(DEVICE)
        for ep in range(epochs):
            opt.zero_grad(); crit(self.head(self.proj(t_X)), t_y).backward(); opt.step()
        with torch.no_grad(): z = self.proj(t_X).cpu().numpy()
        z_pca = self.pca.fit_transform(z).astype(np.float32)
        self.mu = z_pca.mean(0, keepdims=True); self.sd = z_pca.std(0, keepdims=True) + 1e-8
        return self
    def transform(self, X):
        with torch.no_grad(): z = self.proj(torch.from_numpy(X).float().to(DEVICE)).cpu().numpy()
        return (self.pca.transform(z).astype(np.float32) - self.mu) / self.sd

## Ejecución del Benchmark Multi-Backbone: 12 Combinaciones


In [ ]:
benchmark_results = []
roc_curves_data = {}

for b in BACKBONES:
    data = embeddings_data[b]
    X_tr, y_tr = data["X_tr"], data["y_tr"]
    X_va, y_va = data["X_va"], data["y_va"]
    X_te, y_te = data["X_te"], data["y_te"]
    
    in_dim = X_tr.shape[1]
    mu = X_tr.mean(0, keepdims=True); sd = X_tr.std(0, keepdims=True) + 1e-8
    X_tr_z = (X_tr - mu)/sd; X_va_z = (X_va - mu)/sd; X_te_z = (X_te - mu)/sd
    
    print(f"\n{'#'*60}\n  PROCESANDO BACKBONE: {b.upper()} (D={in_dim})\n{'#'*60}")
    
    for m in METHODS:
        t0 = time.time()
        if m == 'PCA': red = PCAReducer(out_dim=8).fit(X_tr_z)
        elif m == 'Linear': red = LinearBottleneckReducer(in_dim, out_dim=8).fit(X_tr_z, y_tr)
        elif m == 'S-PCA': red = SupervisedPCAReducer(in_dim, out_dim=8).fit(X_tr_z, y_tr)
        
        z_tr = red.transform(X_tr_z); z_va = red.transform(X_va_z); z_te = red.transform(X_te_z)
        
        symb = SymbHead(in_dim=8, hidden=16, grid=3, k=3, seed=42, device=DEVICE)
        auc_val, auc_test_full = symb.fit_and_sparsify(
            torch.from_numpy(z_tr).float().to(DEVICE), torch.from_numpy(y_tr).long().to(DEVICE),
            torch.from_numpy(z_va).float().to(DEVICE), torch.from_numpy(y_va).long().to(DEVICE),
            torch.from_numpy(z_te).float().to(DEVICE), torch.from_numpy(y_te).long().to(DEVICE),
            fit_ep=10, sparse_ep=20
        )
        D_sw, kept, _ = symb.prune_sweep(torch.from_numpy(z_va).float().to(DEVICE), torch.from_numpy(z_te).float().to(DEVICE),
                                         y_va, y_te, 0.70, dims=(8, 6, 4, 2))
        kp = symb.build_pruned(kept)
        with torch.no_grad(): kp(torch.from_numpy(z_va).float().to(DEVICE))
        kp.symbolic_enabled = True
        symb.fix_layer1(kp); symb.fix_layer0(kp, top_k=12, x_val=torch.from_numpy(z_va).float().to(DEVICE))
        kp.symbolic_enabled = False
        fben, fmal = symb.make_formula(kp)
        s_test = SymbHead.eval_formula_vec(fmal, z_te[:, kept]) - SymbHead.eval_formula_vec(fben, z_te[:, kept])
        auc_d = float(roc_auc_score(y_te, s_test))
        fpr, tpr, _ = roc_curve(y_te, s_test)
        
        form_str = f'({fmal}) - ({fben})'
        used_vars = sorted(list(set(re.findall(r'x_(\d+)', form_str))))
        n_vars = len(used_vars)
        var_names = [f"x{kept[int(v)-1]}" for v in used_vars]
        dt = time.time() - t0
        
        print(f"--> {b:15s} | {m:7s} | AUC_Full = {auc_test_full:.4f} | AUC_D = {auc_d:.4f} | Vars = {n_vars} {var_names} | Tiempo: {dt:.1f}s")
        benchmark_results.append({
            "Backbone": b, "D_in": in_dim, "Método": m, "AUC_Full": round(float(auc_test_full), 4),
            "AUC_D": round(float(auc_d), 4), "N_Vars": n_vars, "Variables": f"{{{', '.join(var_names)}}}",
            "Longitud_Fórmula": len(form_str), "Tiempo_s": round(dt, 2)
        })
        roc_curves_data[f"{b}_{m}"] = {"fpr": fpr.tolist(), "tpr": tpr.tolist(), "auc": auc_d}


############################################################
  PROCESANDO BACKBONE: RESNET18 (D=512)
############################################################
--> resnet18        | PCA     | AUC_Full = 0.7199 | AUC_D = 0.7301 | Vars = 2 ['x0', 'x1'] | Tiempo: 6.1s
--> resnet18        | Linear  | AUC_Full = 0.7267 | AUC_D = 0.7269 | Vars = 5 ['x1', 'x3', 'x4', 'x6', 'x7'] | Tiempo: 5.6s
--> resnet18        | S-PCA   | AUC_Full = 0.7171 | AUC_D = 0.7273 | Vars = 1 ['x0'] | Tiempo: 5.6s

############################################################
  PROCESANDO BACKBONE: DENSENET121 (D=1024)
############################################################
--> densenet121     | PCA     | AUC_Full = 0.5918 | AUC_D = 0.5631 | Vars = 2 ['x1', 'x6'] | Tiempo: 5.5s
--> densenet121     | Linear  | AUC_Full = 0.6435 | AUC_D = 0.6321 | Vars = 2 ['x2', 'x7'] | Tiempo: 5.5s
--> densenet121     | S-PCA   | AUC_Full = 0.6708 | AUC_D = 0.6801 | Vars = 2 ['x0', 'x3'] | Tiempo: 5.6s

#####################

In [ ]:
df_bench = pd.DataFrame(benchmark_results)
print("\n" + "="*80)
print("  TABLA RESUMEN COMPARATIVA: REDUCCIÓN DIMENSIONAL EN 4 BACKBONES")
print("="*80)
print(df_bench[["Backbone", "D_in", "Método", "AUC_Full", "AUC_D", "N_Vars", "Variables", "Longitud_Fórmula", "Tiempo_s"]].to_string(index=False))


  TABLA RESUMEN COMPARATIVA: REDUCCIÓN DIMENSIONAL EN 4 BACKBONES
       Backbone  D_in Método  AUC_Full  AUC_D  N_Vars            Variables  Longitud_Fórmula  Tiempo_s
       resnet18   512    PCA    0.7199 0.7301       2             {x0, x1}               944      6.14
       resnet18   512 Linear    0.7267 0.7269       5 {x1, x3, x4, x6, x7}              1253      5.64
       resnet18   512  S-PCA    0.7171 0.7273       1                 {x0}               787      5.63
    densenet121  1024    PCA    0.5918 0.5631       2             {x1, x6}               954      5.52
    densenet121  1024 Linear    0.6435 0.6321       2             {x2, x7}               882      5.50
    densenet121  1024  S-PCA    0.6708 0.6801       2             {x0, x3}               811      5.57
efficientnet_b0  1280    PCA    0.5676 0.5684       2             {x1, x5}               742      5.54
efficientnet_b0  1280 Linear    0.6437 0.6120       2             {x1, x3}               844      5.48
effici

In [ ]:
# Gráfica de Barras Comparativa (AUC_D vs N_Vars)
plt.figure(figsize=(14, 5.5))
plt.subplot(1, 2, 1)
x = np.arange(len(BACKBONES))
width = 0.25
colors = ["#d95f02", "#7570b3", "#1b9e77"]

for i, m_name in enumerate(METHODS):
    sub = df_bench[df_bench["Método"] == m_name]
    plt.bar(x + (i - 1) * width, sub["AUC_D"], width=width, label=m_name, color=colors[i], alpha=0.85)

plt.axhline(0.70, color="gray", linestyle="--", alpha=0.7, label="Umbral AUC 0.70")
plt.xticks(x, [f"{b}\n({df_bench[df_bench['Backbone']==b]['D_in'].iloc[0]}D)" for b in BACKBONES], fontsize=10)
plt.ylabel("AUC de la Fórmula Simbólica D(x)", fontsize=11)
plt.title("A) Poder Predictivo de la Fórmula Simbólica (AUC_D)", fontsize=11, fontweight="bold")
plt.ylim(0.50, 0.76)
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(frameon=True, facecolor="white")

plt.subplot(1, 2, 2)
for i, m_name in enumerate(METHODS):
    sub = df_bench[df_bench["Método"] == m_name]
    plt.bar(x + (i - 1) * width, sub["N_Vars"], width=width, label=m_name, color=colors[i], alpha=0.85)

plt.xticks(x, [f"{b}\n({df_bench[df_bench['Backbone']==b]['D_in'].iloc[0]}D)" for b in BACKBONES], fontsize=10)
plt.ylabel("Número de Variables Activas", fontsize=11)
plt.title("B) Compacidad Algebraica (Menor es Más Interpretable)", fontsize=11, fontweight="bold")
plt.ylim(0, 6)
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(frameon=True, facecolor="white")

plt.tight_layout()
plt.show()

In [ ]:
# Curvas ROC Comparativas
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()
for idx, b_name in enumerate(BACKBONES):
    ax = axes[idx]
    dim = df_bench[df_bench["Backbone"] == b_name]["D_in"].iloc[0]
    for i, m_name in enumerate(METHODS):
        key = f"{b_name}_{m_name}"
        data = roc_curves_data[key]
        ax.plot(data["fpr"], data["tpr"], lw=2, color=colors[i],
                label=f"{m_name} (AUC = {data['auc']:.4f})")
    ax.plot([0, 1], [0, 1], color="gray", linestyle="--", lw=1)
    ax.set_title(f"{b_name} (D = {dim})", fontsize=11, fontweight="bold")
    ax.set_xlabel("1 - Especificidad")
    ax.set_ylabel("Sensibilidad")
    ax.grid(True, linestyle=":", alpha=0.6)
    ax.legend(loc="lower right", frameon=True, facecolor="white", framealpha=0.9)

plt.suptitle("Curvas ROC de Fórmulas Simbólicas: PCA vs. Linear vs. S-PCA", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## Cierre

1. D=512:
   Tanto `PCA`, `Linear` como `S-PCA` alcanzan un rendimiento sobresaliente ($	ext{AUC}_D \sim 0.73$). Sin embargo, `S-PCA` es el único que concentra la decisión en 1 sola variable ($x_0 = PC_0$), mientras que `Linear` requiere 5 variables activas.
2. D>=1024:
   - PCA: Colapsa en DenseNet-121 ($	ext{AUC}_D = 0.5631$), EfficientNet-B0 ($0.5684$) y ResNet-50 ($0.5925$).
   - Linear: Recupera capacidad predictiva pero genera dispersión en variables y menor AUC ($\sim 0.56 - 0.63$).
   - `S-PCA` (Supervised-PCA): Supera a todos los métodos en todas las arquitecturas ($	ext{AUC}_D = 0.6801$ en DenseNet-121, $0.6510$ en EfficientNet-B0 y $0.6788$ en ResNet-50) manteniendo fórmulas ultra-compactas de solo 1 o 2 variables.
3. Lectura:
   En espacios profundos no basta con maximizar la varianza total (PCA ciego); hay que alinear primero el subespacio con supervisión ($W$) y luego aplicar descomposición espectral ($V$).
